# 04 — Production monitoring and evaluation\n\nThis notebook simulates production monitoring: drift detection results, retraining triggers, and pipeline reliability metrics. It demonstrates how the pipeline responds to shifting data distributions.

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_DIR)

from src.data_loader import (
    load_training_data, load_drift_data,
    run_all_validations, generate_synthetic_churn_data,
)
from src.model import (
    ModelTrainer, ModelEvaluator, ModelRegistry,
    compute_psi, compare_models, MetricsLogger,
)

In [ ]:
data_dir = os.path.join(PROJECT_DIR, "data")
train_df, test_df = load_training_data(data_dir)
drift_df = load_drift_data(data_dir)

# Train baseline model
trainer = ModelTrainer()
trainer.fit(train_df)
baseline_metrics = ModelEvaluator.evaluate(trainer, test_df)
print("Baseline metrics on clean test set:")
for k, v in baseline_metrics.items():
    print(f"  {k}: {v:.4f}")

## Performance on drifted data\n\nEvaluate the production model on the drift dataset to see how performance degrades when data distributions shift.

In [ ]:
drift_metrics = ModelEvaluator.evaluate(trainer, drift_df)
print("Metrics on drifted data:")
for k, v in drift_metrics.items():
    print(f"  {k}: {v:.4f}")

print()
print("Performance degradation:")
for k in baseline_metrics:
    diff = drift_metrics[k] - baseline_metrics[k]
    print(f"  {k}: {diff:+.4f}")

In [ ]:
# Visualize degradation
metric_names = list(baseline_metrics.keys())
base_vals = [baseline_metrics[m] for m in metric_names]
drift_vals = [drift_metrics[m] for m in metric_names]

x = np.arange(len(metric_names))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w/2, base_vals, w, label="Clean test set", color="#3B6FD4")
ax.bar(x + w/2, drift_vals, w, label="Drifted data", color="#E8C230")
ax.set_xticks(x)
ax.set_xticklabels(metric_names)
ax.set_ylabel("Score")
ax.set_title("Model performance — clean vs drifted data")
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## Drift detection results\n\nPSI scores across all numeric features, comparing training baseline against the drift dataset.

In [ ]:
psi_scores = compute_psi(train_df, drift_df)
psi_df = pd.DataFrame([
    {"Feature": k, "PSI": round(v, 4),
     "Status": "Retrain" if v > 0.25 else ("Investigate" if v > 0.10 else "Stable")}
    for k, v in sorted(psi_scores.items(), key=lambda x: -x[1])
])

max_psi = psi_df["PSI"].max()
n_retrain = (psi_df["Status"] == "Retrain").sum()
n_investigate = (psi_df["Status"] == "Investigate").sum()

print(f"Max PSI: {max_psi:.4f}")
print(f"Features above retrain threshold (0.25): {n_retrain}")
print(f"Features above investigation threshold (0.10): {n_investigate}")
print()
psi_df

In [ ]:
colors_map = {"Retrain": "#ef4444", "Investigate": "#f59e0b", "Stable": "#22c55e"}
colors = [colors_map[s] for s in psi_df["Status"]]

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(psi_df["Feature"], psi_df["PSI"], color=colors)
ax.axvline(x=0.25, color="#ef4444", linestyle="--", linewidth=1.5, label="Retrain (0.25)")
ax.axvline(x=0.10, color="#f59e0b", linestyle=":", linewidth=1.5, label="Investigate (0.10)")
ax.set_xlabel("PSI")
ax.set_title("Drift detection — PSI per feature")
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Simulated retraining trigger\n\nWhen PSI exceeds the threshold, the pipeline recommends retraining. Let us simulate retraining on a mix of old and new data and check if performance recovers.

In [ ]:
# Retrain on combined data (old + drift)
retrain_df = pd.concat([train_df, drift_df], ignore_index=True)
retrained = ModelTrainer()
retrained.fit(retrain_df)

retrained_metrics = ModelEvaluator.evaluate(retrained, drift_df)
print("Retrained model metrics on drift data:")
for k, v in retrained_metrics.items():
    print(f"  {k}: {v:.4f}")

print()
print("Recovery vs original model on drift data:")
for k in drift_metrics:
    improvement = retrained_metrics[k] - drift_metrics[k]
    print(f"  {k}: {improvement:+.4f}")

## Simulated drift over time\n\nGenerate a series of datasets with progressively increasing drift to show how PSI and model performance evolve.

In [ ]:
# Simulate gradual drift by mixing clean and drifted data in varying proportions
mix_ratios = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
results = []

clean_data = generate_synthetic_churn_data(2000, seed=200, drift=False)
drifted_data = generate_synthetic_churn_data(2000, seed=201, drift=True)

for ratio in mix_ratios:
    n_drift = int(2000 * ratio)
    n_clean = 2000 - n_drift
    mixed = pd.concat([
        clean_data.sample(n=n_clean, random_state=42),
        drifted_data.sample(n=n_drift, random_state=42),
    ], ignore_index=True)
    
    psi = compute_psi(train_df, mixed)
    max_psi_val = max(psi.values()) if psi else 0
    m = ModelEvaluator.evaluate(trainer, mixed)
    
    results.append({
        "drift_ratio": ratio,
        "max_psi": max_psi_val,
        "roc_auc": m["roc_auc"],
        "f1": m["f1"],
    })

sim_df = pd.DataFrame(results)
sim_df

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.plot(sim_df["drift_ratio"], sim_df["max_psi"], "o-", color="#E8C230", label="Max PSI", linewidth=2)
ax1.axhline(y=0.25, color="#ef4444", linestyle="--", alpha=0.5, label="Retrain threshold")
ax1.set_xlabel("Drift ratio (proportion of drifted data)")
ax1.set_ylabel("Max PSI", color="#E8C230")
ax1.tick_params(axis="y", labelcolor="#E8C230")

ax2.plot(sim_df["drift_ratio"], sim_df["roc_auc"], "s-", color="#3B6FD4", label="ROC AUC", linewidth=2)
ax2.plot(sim_df["drift_ratio"], sim_df["f1"], "^-", color="#22c55e", label="F1", linewidth=2)
ax2.set_ylabel("Model performance")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower left")

ax1.set_title("Drift progression — PSI and model performance")
plt.tight_layout()
plt.show()

## Pipeline reliability metrics\n\nSummarize the data validation pass rate, drift detection accuracy, and promotion decisions across simulated runs.

In [ ]:
# Simulate multiple pipeline runs with different data conditions
run_results = []
for i, seed in enumerate([42, 55, 77, 88, 100, 123, 150, 200]):
    is_drift = seed in [77, 88, 150]
    data = generate_synthetic_churn_data(1500, seed=seed, drift=is_drift)
    
    validation = run_all_validations(data)
    psi = compute_psi(train_df, data)
    max_psi_val = max(psi.values()) if psi else 0
    m = ModelEvaluator.evaluate(trainer, data)
    
    run_results.append({
        "Run": i + 1,
        "Seed": seed,
        "Has drift": is_drift,
        "Validation passed": validation.passed,
        "Max PSI": round(max_psi_val, 4),
        "Drift detected (PSI > 0.25)": max_psi_val > 0.25,
        "Correct detection": (max_psi_val > 0.25) == is_drift,
        "AUC": round(m["roc_auc"], 4),
    })

run_df = pd.DataFrame(run_results)
run_df

In [ ]:
detection_accuracy = run_df["Correct detection"].mean()
validation_pass_rate = run_df["Validation passed"].mean()

print(f"Drift detection accuracy: {detection_accuracy:.0%}")
print(f"Validation pass rate:     {validation_pass_rate:.0%}")
print(f"Average AUC (clean):      {run_df[~run_df['Has drift']]['AUC'].mean():.4f}")
print(f"Average AUC (drifted):    {run_df[run_df['Has drift']]['AUC'].mean():.4f}")

In [ ]:
# Run summary chart
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# PSI by run
colors_run = ["#ef4444" if d else "#3B6FD4" for d in run_df["Has drift"]]
axes[0].bar(run_df["Run"].astype(str), run_df["Max PSI"], color=colors_run)
axes[0].axhline(y=0.25, color="#ef4444", linestyle="--", alpha=0.5)
axes[0].set_xlabel("Run")
axes[0].set_ylabel("Max PSI")
axes[0].set_title("Max PSI per run (red = drifted data)")

# AUC by run
axes[1].bar(run_df["Run"].astype(str), run_df["AUC"], color=colors_run)
axes[1].set_xlabel("Run")
axes[1].set_ylabel("ROC AUC")
axes[1].set_title("AUC per run (red = drifted data)")
axes[1].set_ylim(0.5, 1.0)

plt.tight_layout()
plt.show()

## Summary\n\n- The baseline model shows measurable performance degradation on drifted data, confirming the need for monitoring\n- PSI reliably detects drift in the simulated datasets and correctly triggers retraining recommendations\n- Retraining on combined (old + drifted) data recovers performance on the shifted distribution\n- The gradual drift simulation shows that PSI increases proportionally with the drift ratio, providing early warning\n- Across 8 simulated runs, the pipeline correctly identifies drift conditions with high accuracy\n- The full pipeline (validate, train, evaluate, compare, promote) runs end-to-end without manual intervention